In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import sys

# The notebook runs from notebooks/, so we climb one level to the project root
# and add it to sys.path so "src.support..." becomes importable.
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.support.parsing_utils import get_nested, classify, parse_filename
from src.support.EDA_utils import eda_report, flag_suspicious_zeros, summary_stats

RAW_DIR = PROJECT_ROOT / "data" / "raw"

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np  

In [2]:
FIELDS = [
    "id", "title", "description", "created",
    "company.display_name",
    "location.display_name", "location.area",
    "category.label", "category.tag",
    "salary_min", "salary_max", "salary_is_predicted",
    "contract_type", "contract_time",
    "latitude", "longitude", "redirect_url",
]

In [ ]:
counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
offers_per_category = defaultdict(int)
total_offers = 0

for path in sorted(RAW_DIR.glob("adzuna_*.json")):
    category, page = parse_filename(path.name)
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)          # full API response, kept intact
    for offer in payload.get("results", []):   # offers live under "results"
        total_offers += 1
        offers_per_category[category] += 1
        for field in FIELDS:
            state = classify(get_nested(offer, field))
            counts[category][field][state] += 1

print(f"Total offers: {total_offers}")
print(f"Per category: {dict(offers_per_category)}")

Total offers: 951
Per category: {'consultancy-jobs': 250, 'hr-jobs': 250, 'it-jobs': 250, 'legal-jobs': 201}


In [13]:
print(f"{'FIELD':<28} {'present':>8} {'%present_field':>15} {'null':>6} {'empty':>6} {'absent':>7}")
print("-" * 72)
for field in FIELDS:
    present = sum(counts[c][field]["present"] for c in counts)
    null    = sum(counts[c][field]["null"]    for c in counts)
    empty   = sum(counts[c][field]["empty"]   for c in counts)
    absent  = sum(counts[c][field]["absent"]  for c in counts)
    pct = (present / total_offers * 100) if total_offers else 0
    print(f"{field:<28} {present:>8} {pct:>15.2f}% {null:>6} {empty:>6} {absent:>7}")

FIELD                         present  %present_field   null  empty  absent
------------------------------------------------------------------------
id                                951          100.00%      0      0       0
title                             951          100.00%      0      0       0
description                       951          100.00%      0      0       0
created                           951          100.00%      0      0       0
company.display_name              860           90.43%      0      0      91
location.display_name             951          100.00%      0      0       0
location.area                     951          100.00%      0      0       0
category.label                    951          100.00%      0      0       0
category.tag                      951          100.00%      0      0       0
salary_min                        556           58.46%      0      0     395
salary_max                        556           58.46%      0      0     395
sala

In [10]:
# Of the offers WITH a salary, how many are Adzuna estimates vs real?
real = predicted = 0
for path in sorted(RAW_DIR.glob("adzuna_*.json")):
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)
    for offer in payload.get("results", []):
        if "salary_min" in offer:                 # has some salary
            if offer.get("salary_is_predicted") in ("1", 1):
                predicted += 1
            else:
                real += 1

print(f"Real (declared): {real}")
print(f"Predicted by Adzuna: {predicted}")

Real (declared): 556
Predicted by Adzuna: 0


In [11]:
# Inspect the ACTUAL raw values of salary_is_predicted before trusting any count.
# We don't assume the format — we look at it.
values_seen = {}   # value -> how many times it appears
example_offer = None

for path in sorted(RAW_DIR.glob("adzuna_*.json")):
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)
    for offer in payload.get("results", []):
        if "salary_min" in offer:
            v = offer.get("salary_is_predicted")
            # record the value AND its Python type, so "1" (str) vs 1 (int) is visible
            key = f"{repr(v)}  (type: {type(v).__name__})"
            values_seen[key] = values_seen.get(key, 0) + 1
            if example_offer is None:
                example_offer = offer

print("Distinct values of salary_is_predicted (among offers with salary):")
for k, n in values_seen.items():
    print(f"  {k}  ->  {n} offers")

Distinct values of salary_is_predicted (among offers with salary):
  '0'  (type: str)  ->  556 offers


In [12]:
# Look at real examples: is the salary offer-specific or a generic category range?
shown = 0
for path in sorted(RAW_DIR.glob("adzuna_*.json")):
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)
    for offer in payload.get("results", []):
        if "salary_min" in offer and shown < 5:
            print(f"{offer.get('title')[:50]:<52} "
                  f"min={offer.get('salary_min')}  max={offer.get('salary_max')}  "
                  f"cat={offer.get('category', {}).get('label')}")
            shown += 1

Senior Product Manager - Money (Barcelona)           min=65000  max=85000  cat=Trabajos en consultoría
Director, Viral Vector BD — Gene Therapy Partnersh   min=70000  max=90000  cat=Trabajos en consultoría
Director of Assisted Channels Transformation         min=70000  max=90000  cat=Trabajos en consultoría
Global Expansion Director: Market Strategy           min=60000  max=80000  cat=Trabajos en consultoría
Regulatory Strategy Lead – Energy Policy (Remote)    min=80000  max=100000  cat=Trabajos en consultoría


## EDA

<function src.support.EDA_utils.eda_report(df)>